### Auto encoding + K means

In [2]:
import sys
sys.path.append('..')
from scripts.dataImport import fetch_datasets, load_flow, load_packet

fetch_datasets()

Datasets already downloaded.


### Load Datasets
Load only what you need. Available keys: `benign`, `ddos_http`, `dos_http`, `dns_spoofing`, `xss`, `brute_force`

In [22]:
# Example - load what you need for your task
df_xss_flow = load_flow("xss")
df_benign_packet = load_packet("benign")

Loading Flow Dataset: xss...
Shape: (3377, 84)
Loading Packet Dataset: benign...
Shape: (1182798, 135)


In [ ]:
df_benign_packet.head()

,stream,src_mac,dst_mac,src_ip,dst_ip,src_port,dst_port,inter_arrival_time,time_since_previously_displayed_frame,port_class_dst,...,sum_p,min_p,max_p,med_p,average_p,var_p,q3_p,q1_p,iqr_p,l3_ip_dst_count
0,0,Arlo Q Indoor Camera,3c:18:a0:41:c3:a0,192.168.137.175,99.81.244.93,56891,443,0.000000,0.000000,1,...,58160.0,2908.0,2908.0,2908.0,2908.0,0.000000,2908.0,2908.0,0.0,1.0
1,0,Arlo Q Indoor Camera,3c:18:a0:41:c3:a0,192.168.137.175,99.81.244.93,56891,443,0.000164,0.000164,1,...,58160.0,2908.0,2908.0,2908.0,2908.0,0.000000,2908.0,2908.0,0.0,1.0
2,0,Arlo Q Indoor Camera,3c:18:a0:41:c3:a0,192.168.137.175,99.81.244.93,56891,443,0.000269,0.000105,1,...,56712.0,1460.0,2908.0,2908.0,2835.6,104835.200000,2908.0,2908.0,0.0,1.0
3,0,Arlo Q Indoor Camera,3c:18:a0:41:c3:a0,192.168.137.175,99.81.244.93,56891,443,0.000414,0.000145,1,...,55264.0,1460.0,2908.0,2908.0,2763.2,198635.115789,2908.0,2908.0,0.0,1.0
4,0,Arlo Q Indoor Camera,3c:18:a0:41:c3:a0,192.168.137.175,99.81.244.93,56891,443,0.001800,0.001386,1,...,53816.0,1460.0,2908.0,2908.0,2690.8,281399.747368,2908.0,2908.0,0.0,1.0


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from scripts.dataImport import fetch_datasets, load_flow, load_packet


df_benign = load_packet("benign").copy()
df_benign["attack_type_label"] = "benign"
df_benign["is_anomaly"] = 0

attack_types = ["ddos_http", "dos_http", "dns_spoofing", "xss", "brute_force"]
attack_list = []

for attack_type in attack_types:
    try:
        df = load_packet(attack_type).copy()
        df["attack_type_label"] = attack_type
        df["is_anomaly"] = 1
        attack_list.append(df)
        print(f"Loaded {attack_type}: {df.shape}")
    except Exception as e:
        print(f"Failed to load {attack_type}: {e}")

if len(attack_list) == 0:
    raise ValueError("No attack data was loaded.")

df_attack = pd.concat(attack_list, ignore_index=True)

# split benign
df_train_benign, df_test_benign = train_test_split( # should occur after phase2preprocess
    df_benign,
    test_size=0.2,
    random_state=42
)

# Test：20% benign + all attacks
df_test_all = pd.concat( #should occur after phase2.preprocess
    [df_test_benign, df_attack],
    ignore_index=True
)
#I think this can be combined with drop_features. Simplifies later code - Mack
label_cols = ["is_anomaly", "attack_type_label"]

candidate_cols = [# see comment above
    col for col in df_train_benign.columns
    if col not in label_cols
]

numeric_features = ( #added to phase2preprocess
    df_train_benign[candidate_cols]
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)
#We need to do our feature selection tests with this list - Mack
drop_features = [ #added to phase2preproces as variable
    "time_since_previously_displayed_frame",
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port"
]

numeric_features = [ # added to phase2preprocess
    col for col in numeric_features
    if col not in drop_features
]

if len(numeric_features) == 0:# added to phase2preprocess
    raise ValueError("No numeric features found.")

X_train = ( # added to phase2preprocess
    df_train_benign
    .reindex(columns=numeric_features)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype(np.float64)
)

X_test = (# added to phase2preprocess
    df_test_all
    .reindex(columns=numeric_features)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype(np.float64)
)


y_test = df_test_all["is_anomaly"].to_numpy() # should occur after phase2preprocess
y_test_attack_types = df_test_all["attack_type_label"].to_numpy()# should occur after phase2preprocess

train_std = X_train.std(axis=0)# should occur after phase2preprocess
#This is a feature selection choice. Not in scope of phase2preprocess.py
valid_features = train_std[train_std > 1e-8].index.tolist()# should occur after phase2preprocess

removed_features = [f for f in numeric_features if f not in valid_features]# should occur after phase2preprocess
print(f"Numeric features before filtering: {len(numeric_features)}")
print(f"Removed near-constant features: {len(removed_features)}")
print(f"Features used for model: {len(valid_features)}")

if len(valid_features) == 0:
    raise ValueError("All features are constant in benign training data.")

X_train = X_train[valid_features]
X_test = X_test[valid_features]

log_features = [ # added to phase2preprocess. Unsure if in scope
    "stream_jitter_1_var",
]

for col in log_features: # added to phase2preprocess. Unsure if in scope
    if col in X_train.columns:
        
        X_train[col] = np.log1p(X_train[col].clip(lower=0))
        X_test[col] = np.log1p(X_test[col].clip(lower=0))

scaler = StandardScaler()# should occur after phase2preprocess

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# all below should occur after phase2preprocess
CLIP_VALUE = 15.0

print("Max |z-score| before clipping (train):", np.abs(X_train_scaled).max())
print("Max |z-score| before clipping (test): ", np.abs(X_test_scaled).max())

max_z_per_feature = np.abs(X_test_scaled).max(axis=0)
top_idx = np.argsort(max_z_per_feature)[-10:][::-1]

print("\nTop features by test z-score:")
for i in top_idx:
    print(f"{valid_features[i]}: max |z| = {max_z_per_feature[i]:.4f}")

X_train_scaled = np.clip(X_train_scaled, -CLIP_VALUE, CLIP_VALUE)
X_test_scaled = np.clip(X_test_scaled, -CLIP_VALUE, CLIP_VALUE)
# should occur after phase2preprocess
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

print("\nFinal shapes:")
print("Features after filtering:", X_train_tensor.shape[1])
print("Shape of training set:", X_train_tensor.shape)
print("Shape of test set:", X_test_tensor.shape)
print("Shape of test labels:", y_test.shape)
print("Benign test samples:", (y_test == 0).sum())
print("Attack test samples:", (y_test == 1).sum())

col = "time_since_previously_displayed_frame"

print("\nRaw value diagnostics:")
for name, df in [("benign", df_benign)] + [
    (attack_type, attack_list[i])
    for i, attack_type in enumerate(attack_types)
]:
    s = pd.to_numeric(df[col], errors="coerce")
    print(
        name,
        "min=", s.min(),
        "median=", s.median(),
        "p99=", s.quantile(0.99),
        "max=", s.max(),
        "NaN=", s.isna().sum()
    )


Loading Packet Dataset: benign...
Shape: (1182798, 135)
Loading Packet Dataset: ddos_http...
Shape: (290346, 135)
Loaded ddos_http: (290346, 137)
Loading Packet Dataset: dos_http...
Shape: (314039, 135)
Loaded dos_http: (314039, 137)
Loading Packet Dataset: dns_spoofing...
Shape: (296025, 135)
Loaded dns_spoofing: (296025, 137)
Loading Packet Dataset: xss...
Shape: (40101, 135)
Loaded xss: (40101, 137)
Loading Packet Dataset: brute_force...
Shape: (131477, 135)
Loaded brute_force: (131477, 137)
Numeric features before filtering: 116
Removed near-constant features: 1
Features used for model: 115
Max |z-score| before clipping (train): 688.4103970619311
Max |z-score| before clipping (test):  154067.70019289153

Top features by test z-score:
stream_jitter_5_var: max |z| = 154067.7002
stream_jitter_10_var: max |z| = 153965.6696
stream_jitter_30_var: max |z| = 153877.1622
stream_jitter_60_var: max |z| = 153846.8253
ntp_interval: max |z| = 30834.0078
dns_interval: max |z| = 19519.6572
http_co

In [25]:
# Autoencoder model definition

class PacketAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(PacketAutoencoder, self).__init__()
        
        # Encoder：compress the 119-dimensional input down to 8 dimensions
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)  # Latent Feature
        )
        # Decoder: reconstruct the input from the 8-dimensional latent space back to 119 dimensions
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent


input_dim = X_train_tensor.shape[1]  # get 119 features 
model = PacketAutoencoder(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)  # learning rate = 0.003

# Create DataLoader for training
batch_size = 512  # set batch size to 512 for faster training
train_loader = torch.utils.data.DataLoader(X_train_tensor, batch_size=batch_size, shuffle=True)

# train the autoencoder on the benign dataset (don't use the attack data for training)
num_epochs = 15  
print("on training...")

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        reconstructed, _ = model(batch)
        loss = criterion(reconstructed, batch)  # minimize reconstruction loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    
    # Print the loss every 3 epochs and also for the first epoch
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {epoch_loss / len(X_train_tensor):.6f}")

print("finished training the autoencoder on benign data.")

on training...
Epoch [1/15], Reconstruction Loss: 0.127709
Epoch [3/15], Reconstruction Loss: 0.045225
Epoch [6/15], Reconstruction Loss: 0.036106
Epoch [9/15], Reconstruction Loss: 0.032158
Epoch [12/15], Reconstruction Loss: 0.029694
Epoch [15/15], Reconstruction Loss: 0.028517
finished training the autoencoder on benign data.


In [26]:
# k-means clustering on the latent features and reconstruction errors
# use the trained autoencoder to extract latent features and reconstruction errors from the test set
model.eval()
with torch.no_grad():
    # predict the test set (contains both benign and ddos_http attack)
    reconstructed, latent = model(X_test_tensor)

# calculate the mean squared error (MSE) for each packet (row) in the test set
reconstruction_errors = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1).numpy()
latent_features = latent.numpy()

# combine the latent features and reconstruction errors into a single feature set for K-means clustering
kmeans_input = np.hstack((latent_features, reconstruction_errors.reshape(-1, 1)))

print("K-means input shape:", kmeans_input.shape)

# k-means clustering (K=2)
print("on going K-means ...")
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_assignments = kmeans.fit_predict(kmeans_input)

# recognize which cluster is the anomaly cluster based on reconstruction error
# the cluster with significantly higher reconstruction error is considered the anomaly cluster
error_c0 = reconstruction_errors[cluster_assignments == 0].mean()
error_c1 = reconstruction_errors[cluster_assignments == 1].mean()

anomaly_cluster = 1 if error_c1 > error_c0 else 0
print(f"\nCluster 0 average reconstruction error: {error_c0:.6f}")
print(f"Cluster 1 average reconstruction error: {error_c1:.6f}")
print(f"The system classifies Cluster {anomaly_cluster} as the [Anomaly Traffic Class]")

# convert K-means cluster assignments to predicted labels (1 for anomaly, 0 for benign)
y_pred = np.where(cluster_assignments == anomaly_cluster, 1, 0)

# evaluate the unsupervised model's detection performance
print("\n" + "="*20 + " Initial Detection Report " + "="*20)
print(classification_report(y_test, y_pred, target_names=['Benign', 'Anomaly']))

# calculate AUC-ROC
auc = roc_auc_score(y_test, reconstruction_errors)
print(f"Autoencoder Reconstruction Error AUC-ROC Score: {auc:.4f}")

# print confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"True Benign (Predicted as Benign): {cm[0][0]} | False Positive (False alarm): {cm[0][1]}")
print(f"False Negative (missed detection): {cm[1][0]} | True Anomaly (Correctly detected): {cm[1][1]}")

K-means input shape: (1308548, 9)
on going K-means ...

Cluster 0 average reconstruction error: 0.550542
Cluster 1 average reconstruction error: 7.386566
The system classifies Cluster 1 as the [Anomaly Traffic Class]

==================== Initial Detection Report ====================
              precision    recall  f1-score   support

      Benign       0.27      1.00      0.42    236560
     Anomaly       1.00      0.40      0.57   1071988

    accuracy                           0.51   1308548
   macro avg       0.63      0.70      0.50   1308548
weighted avg       0.87      0.51      0.55   1308548

Autoencoder Reconstruction Error AUC-ROC Score: 0.9300

Confusion Matrix:
True Benign (Predicted as Benign): 235401 | False Positive (False alarm): 1159
False Negative (missed detection): 640244 | True Anomaly (Correctly detected): 431744


In [36]:
cluster_assignments.shape

(1308548,)

In [35]:
reconstruction_errors.shape

(1308548,)

In [37]:
X_test_tensor.numpy().shape

(1308548, 115)